In [10]:
import pandas as pd
df = pd.read_csv(r'D:\SoW\churn-prediction\churn-prediction-main\data\Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
null_count = df['TotalCharges'].isnull().sum()  # Check how many nulls
print(f"Dropping {null_count} rows with missing TotalCharges.")
df.dropna(inplace=True)  # Remove null rows


Dropping 11 rows with missing TotalCharges.


In [11]:
print(df['TotalCharges'].dtype)
print(df.shape)

float64
(7032, 21)


In [14]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import streamlit as st
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split

# --- STEP 1: DATA LOADING & CLEANING ---
try:
    # Use the path relative to your script location
    df = pd.read_csv('../data/Telco-Customer-Churn.csv')
    
    # 1. Handle TotalCharges (The "Problem Child" column)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df.dropna(subset=['TotalCharges'], inplace=True)
    
    # 2. Encode Target (Essential for ROC curve)
    df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)
    
    # 3. Simple Feature Selection (Only numeric for this example to avoid encoding errors)
    # In a full app, you would use a ColumnTransformer here.
    X = df[['tenure', 'MonthlyCharges', 'TotalCharges']]
    y = df['Churn']
    
except FileNotFoundError:
    st.error("CSV not found! Check: ../data/Telco-Customer-Churn.csv")
    st.stop() # Stops execution if data is missing

# --- STEP 2: TRAIN-TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- STEP 3: MODEL TRAINING & SAVING (The "Fix") ---
# We train it here to ensure 'final_model' exists in YOUR version of scikit-learn
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_train, y_train)

# Save it so Step 4 can load it (This fixes your version mismatch)
joblib.dump(final_model, '../src/model_pipeline.joblib')

# --- STEP 4: MODEL LOADING & PREDICTION ---
try:
    # Load the model we just saved
    model = joblib.load('../src/model_pipeline.joblib')
    
    # Get probabilities for the positive class (Churn)
    y_probs = model.predict_proba(X_test)[:, 1]
    
    # Calculate ROC metrics
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)
    
except Exception as e:
    st.error(f"Error during model inference: {e}")
    st.stop()

# --- STEP 5: STREAMLIT UI ---
st.set_page_config(page_title="Mahi Fashion Analytics", page_icon="📊")
st.title("Mahi Fashion: Churn Prediction Analytics")

# Display metrics in a clean row
col1, col2 = st.columns(2)
with col1:
    st.metric("Model Performance (AUC)", f"{roc_auc:.2f}")
with col2:
    st.write("The AUC represents the model's ability to distinguish between customers who stay and those who leave.")

# Plotting the ROC Curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
ax.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--') # Diagonal baseline
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic (ROC)')
ax.legend(loc="lower right")
ax.grid(alpha=0.2)

st.pyplot(fig)

2026-03-31 14:53:19.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.088 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.102 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 14:53:19.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()